In [1]:
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine
from pathlib import Path

/home/vedansh-kakkar/anaconda3/envs/torch130/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/vedansh-kakkar/anaconda3/envs/torch130/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/vedansh-kakkar/anaconda3/envs/torch130/lib/python3.11/site-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layer

In [2]:
# Jupyter starts in the notebook's folder (efficientAD/), not the repo root,
# so resolve the data path from the repo root instead of the working directory.
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "anomaly_folder").is_dir())
root = repo_root / "data" / "anomaly_folder"

In [8]:
dm_val = Folder(
    name="anomaly_folder",
    root=root,
    normal_dir='train/good',
    normal_test_dir='val/good',
    abnormal_dir='val/defect',
    mask_dir='val/mask',
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=8,
    val_split_mode='same_as_test',
)

In [9]:
dm_test = Folder(
    name="anomaly_folder",
    root=root,
    normal_dir="train/good",
    normal_test_dir="test/good",
    abnormal_dir="test/defect",
    mask_dir="test/mask",
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=8,
    # Default (FROM_TEST) would carve half of the test images off as an unused val split.
    val_split_mode='same_as_test',
)

In [10]:
model = Patchcore(
    backbone="wide_resnet50_2",
    layers=("layer2", "layer3"),
    coreset_sampling_ratio=0.1,      # fraction of patch features kept in the memory bank
    num_neighbors=9,                 # k for nearest-neighbor scoring
)

In [11]:
from lightning.pytorch.callbacks import TQDMProgressBar

# Lightning's default Rich progress bar hijacks stdout/stderr; in Jupyter, Patchcore's
# tqdm coreset bar then makes rich and ipykernel flush each other until RecursionError.
engine = Engine(accelerator="gpu", devices=1, callbacks=[TQDMProgressBar()])

In [12]:
engine.fit(model=model, datamodule=dm_val)
engine.test(model=model, datamodule=dm_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/vedansh-kakkar/anaconda3/envs/torch130/lib/python3.11/site-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99.450                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

/home/vedansh-kakkar/anaconda3/envs/torch130/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/vedansh-kakkar/anaconda3/envs/torch130/lib/python3.11/site-packages/lightning/pytorch/loops/fit_loop.py:538: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.


Epoch 0: 100%|██████████| 30/30 [00:02<00:00, 14.66it/s]
Validation: |          | 0/? [00:00<?, ?it/s]

...

Testing DataLoader 0:  40%|████      | 6/15 [00:05<00:08,  1.10it/s]

/home/vedansh-kakkar/anaconda3/envs/torch130/lib/python3.11/site-packages/PIL/Image.py:1039: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Testing DataLoader 0: 100%|██████████| 15/15 [00:16<00:00,  0.90it/s]


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │     0.830912709236145     │
│       image_F1Score       │    0.9165562987327576     │
│        pixel_AUROC        │    0.8363451957702637     │
│       pixel_F1Score       │    0.6032978892326355     │
└───────────────────────────┴───────────────────────────┘

[{'image_AUROC': 0.830912709236145,
  'image_F1Score': 0.9165562987327576,
  'pixel_AUROC': 0.8363451957702637,
  'pixel_F1Score': 0.6032978892326355}]